# Customer Segmentation Analysis (RFM + K-Means)
**Oasis Infobyte SIP · Data Analytics · Level 1 · Task 2**

**Objective:** segment an e-commerce company's customers into distinct groups based on purchasing behaviour (Recency, Frequency, Monetary value) using K-Means clustering, and recommend a marketing action for each segment.

**Dataset:** `online_retail.csv` - transaction lines with InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country (same schema as the UCI *Online Retail* dataset).

> The file included here is generated by `generate_data.py`. To use the real UCI/Kaggle *Online Retail* file, save it as `online_retail.csv` (same column names) and re-run; then update the written observations to match your results.

---
## 1. Setup and data loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("images", exist_ok=True)

df = pd.read_csv("online_retail.csv", parse_dates=["InvoiceDate"])
print("Shape:", df.shape)
df.head()

## 2. Data inspection: missing values and inconsistent data

In [ ]:
print("Null values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Cancelled invoices (InvoiceNo starts with 'C'):", df["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Rows with Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Rows with UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())
print("\nDate range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

**Observation:** the raw data has 37,149 rows. Three quality problems need fixing before analysis: **1,462 rows with no CustomerID** (we cannot segment anonymous buyers), **373 duplicate rows**, and **721 cancelled invoices** with negative quantities (returns, not purchases). Dates run from January to December 2023.

### Cleaning

In [ ]:
before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
df["CustomerID"] = df["CustomerID"].astype(int)
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(f"Rows before: {before:,}  |  after: {len(df):,}  |  removed: {before - len(df):,}")
print("Unique customers:", df["CustomerID"].nunique())
print("Unique invoices:", df["InvoiceNo"].nunique())

**Observation:** after removing duplicates, anonymous rows, cancellations and invalid values we keep **34,612 rows from 499 customers** across 5,164 invoices. The data is now ready for customer-level analysis.

## 3. Descriptive statistics: purchase value, frequency and customer lifetime value

In [ ]:
total_revenue = df["Revenue"].sum()
n_orders = df["InvoiceNo"].nunique()
print(f"Total revenue: {total_revenue:,.0f}")
print(f"Average order value (AOV): {total_revenue / n_orders:,.1f}")

snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
customer = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
)
# Customer Lifetime Value (historical) = total revenue per customer over the period
customer["CLV"] = customer["Monetary"]

summary = customer[["Recency", "Frequency", "Monetary"]].agg(["mean", "median", "std", "min", "max"]).T.round(1)
summary.loc["CLV (historical)"] = customer["CLV"].agg(["mean", "median", "std", "min", "max"]).round(1).values
print()
summary

**Observation:** the average order is worth about **232**, and a typical customer places about **10 orders** (median 8) and spends about **2,406** over the year (median 1,347). Mean above median for frequency and spend shows right-skewed behaviour: a smaller group of very active customers pulls the averages up. On average a customer last bought about 124 days before the end of the period.

## 4. Feature selection: RFM analysis
We use the three classic behavioural features:
- **Recency** - days since the customer's last purchase (lower is better)
- **Frequency** - number of distinct orders
- **Monetary** - total money spent

In [ ]:
rfm = customer[["Recency", "Frequency", "Monetary"]].copy()
print("Skewness:\n", rfm.skew().round(2))

plt.figure(figsize=(14, 4))
for i, col in enumerate(rfm.columns, 1):
    plt.subplot(1, 3, i)
    sns.histplot(rfm[col], kde=True, bins=30)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.savefig("images/01_rfm_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 4))
sns.heatmap(rfm.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("RFM Correlation")
plt.savefig("images/02_rfm_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** all three features are right-skewed (skewness 0.4 for Recency, 1.4 for Frequency, 1.8 for Monetary), so a log transform helps before clustering. **Frequency and Monetary are almost perfectly correlated (0.98)**: customers who order often spend more. Recency is negatively correlated with both (-0.54): customers who bought recently tend to buy more often and spend more.

## 5. Normalisation (StandardScaler)

In [ ]:
# log1p reduces skew; StandardScaler then puts all features on the same scale
rfm_log = np.log1p(rfm)
scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)
print("Scaled mean (~0):", X.mean(axis=0).round(3))
print("Scaled std  (~1):", X.std(axis=0).round(3))

**Why scale?** K-Means uses distances. Without scaling, Monetary (values in the thousands) would dominate Recency and Frequency. After the log transform and standardisation every feature has mean 0 and standard deviation 1.

## 6. K-Means clustering: Elbow Method to choose K

In [ ]:
ks = range(2, 9)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(list(ks), inertia, marker="o")
ax[0].axvline(4, color="red", linestyle="--", label="Chosen K = 4")
ax[0].set_title("Elbow Method (Inertia vs K)"); ax[0].set_xlabel("Number of clusters (K)"); ax[0].set_ylabel("Inertia"); ax[0].legend()
ax[1].plot(list(ks), sil, marker="o", color="green")
ax[1].set_title("Silhouette Score vs K"); ax[1].set_xlabel("Number of clusters (K)"); ax[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.savefig("images/03_elbow_silhouette.png", dpi=150, bbox_inches="tight")
plt.show()
pd.DataFrame({"K": list(ks), "Inertia": np.round(inertia, 1), "Silhouette": np.round(sil, 3)})

**Observation:** inertia drops steeply from K=2 (665) to K=3 (380) and K=4 (283), then flattens (K=5: 212, K=6: 171), so the elbow is around **K=3-4**. The silhouette score is highest at K=2 (0.48) but that would only split customers into "active" and "inactive"; for K=3 to K=6 it stays similar (about 0.46-0.47), so extra clusters do not hurt cluster quality. We choose **K = 4** because it gives four clearly different, actionable business segments.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X)

# Name clusters by average spend (rank-based, so names stay correct on any run)
order = rfm.groupby("Cluster")["Monetary"].mean().sort_values(ascending=False).index
names = dict(zip(order, ["Champions", "Loyal Customers", "At-Risk", "Low-Value / New"]))
rfm["Segment"] = rfm["Cluster"].map(names)
seg_order = ["Champions", "Loyal Customers", "At-Risk", "Low-Value / New"]
print(rfm["Segment"].value_counts())

## 7. Cluster visualisation (scatter plots)

In [ ]:
palette = dict(zip(seg_order, ["#16A34A", "#2563EB", "#EA580C", "#6B7280"]))
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
pairs = [("Recency", "Frequency"), ("Frequency", "Monetary"), ("Recency", "Monetary")]
for a, (x, y) in zip(ax, pairs):
    sns.scatterplot(data=rfm, x=x, y=y, hue="Segment", hue_order=seg_order, palette=palette, alpha=0.8, ax=a)
    a.set_title(f"{x} vs {y}")
plt.tight_layout()
plt.savefig("images/04_cluster_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** the clusters separate cleanly. **Champions** sit at low Recency (bought very recently), high Frequency and the highest Monetary values. **At-Risk** customers have high Recency (they have not bought for months) even though they used to buy fairly often. **Low-Value / New** customers cluster at low Frequency and low spend. **Loyal Customers** sit in the middle: recent, regular, moderate spend.

## 8. Cluster profiling

In [ ]:
profile = rfm.groupby("Segment").agg(
    Customers=("Recency", "size"),
    Avg_Recency_days=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"),
).reindex(seg_order)
profile["Customer_%"] = profile["Customers"] / profile["Customers"].sum() * 100
profile["Revenue_%"] = profile["Total_Revenue"] / profile["Total_Revenue"].sum() * 100
profile["Avg_Order_Value"] = profile["Total_Revenue"] / (profile["Avg_Frequency"] * profile["Customers"])
profile.round(1)

**Observation (cluster profiles):**

| Segment | Customers | Avg recency | Avg orders | Avg spend | Share of customers | Share of revenue |
|---|---|---|---|---|---|---|
| Champions | 82 | 15 days | 28.1 | 8,322 | 16% | **57%** |
| Loyal Customers | 118 | 53 days | 12.8 | 2,653 | 24% | 26% |
| At-Risk | 129 | 222 days | 7.8 | 1,317 | 26% | 14% |
| Low-Value / New | 170 | 151 days | 2.0 | 207 | 34% | 3% |

Champions are only 16% of customers but generate **57% of revenue** (and also have the highest average order value, about 296). The largest group, Low-Value / New, is 34% of customers but only 3% of revenue.

## 9. Number of customers per cluster (bar chart)

In [ ]:
counts = rfm["Segment"].value_counts().reindex(seg_order)
rev_share = profile["Revenue_%"]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(counts.index, counts.values, color=[palette[s] for s in counts.index])
ax[0].set_title("Number of Customers per Segment"); ax[0].set_ylabel("Customers")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[1].bar(rev_share.index, rev_share.values, color=[palette[s] for s in rev_share.index])
ax[1].set_title("Share of Total Revenue per Segment (%)"); ax[1].set_ylabel("% of revenue")
for i, v in enumerate(rev_share.values):
    ax[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom")
for a in ax:
    a.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("images/05_segment_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

rfm.to_csv("rfm_customer_segments.csv")

**Observation:** the customer-count chart and the revenue chart tell opposite stories. Low-Value / New is the largest segment (170 customers) but the smallest revenue contributor (2.9%), while Champions (82 customers) bring in 56.8% of revenue. Revenue is highly concentrated, so losing a few Champions would hurt far more than losing many Low-Value customers.

## 10. Insights and marketing recommendations

| Segment | Who they are | Recommended marketing action |
|---|---|---|
| **Champions** (16% of customers, 57% of revenue) | Bought in the last ~2 weeks, ~28 orders, highest spend | **Reward and retain.** VIP programme, early access to new products, exclusive perks and referral rewards. Avoid heavy discounts: they already buy. |
| **Loyal Customers** (24%, 26%) | Regular buyers, ~13 orders, moderate spend | **Grow their value.** Cross-sell and upsell bundles, loyalty points and "spend X, get Y" offers to move them toward Champion level. |
| **At-Risk** (26%, 14%) | Used to buy ~8 times but silent for ~7 months | **Win back.** Personalised "we miss you" emails with a time-limited offer, product recommendations based on past purchases, and a short feedback survey to learn why they left. |
| **Low-Value / New** (34%, 3%) | ~2 orders, low spend | **Nurture cheaply.** Automated welcome series, a second-purchase incentive and low-cost email campaigns. Do not spend a large budget per customer here. |

**Overall:** put the biggest retention effort on the 16% of customers who produce more than half of revenue, run a win-back campaign for the At-Risk group (the best recovery opportunity, since they have already shown repeat-buying behaviour), and automate low-cost nurturing for the rest.